# Predicting simulated Curie-Temperatures from compound embeddings 

This pipeline trains machine learning models that predict simulated Curie temperatures (Tc_exp, in Kelvin) directly from stoichiometric compound embeddings.

In [ ]:
import sys
from pathlib import Path
import os

# Get project root (parent of notebooks/)
PROJECT_ROOT = Path.cwd().parent.resolve()

# Add project root to Python path so src/ is importable
sys.path.insert(0, str(PROJECT_ROOT))

# Change working directory to project root
os.chdir(PROJECT_ROOT)

### 1. Pre-Process Data

The preprocessing pipeline in this script prepares experimental and simulated Curie temperature (Tc) data.

1. **Aggregate** data from multiple sources.  
2. **Clean** Tc values: remove units, symbols, and uncertainties; convert to float.  
3. **Drop** invalid (non-numeric) Tc entries.  
4. **Deduplicate** by taking the median Tc per composition.  
5. **Flag** compositions containing rare-earth elements.  
6. **Split** data into RE-containing and RE-free subsets.  
7. **Save** clean, structured datasets for analysis.

In [ ]:
from src.process_tc_data import main

In [ ]:
main()

### 2. Embedding Creation

1. **Load Element Embeddings**:  
   Read pre-trained element vectors (e.g., Matscholar200) from a JSON file.

2. **Generate Compound Embeddings**:  
   For each composition, compute a weighted average of its constituent element embeddings, where weights are based on atomic fractions.

3. **Handle Missing Elements**:  
   Skip compounds containing elements not in the embedding dictionary.

4. **Filter Invalid Embeddings**:  
   Remove compounds that couldn’t be embedded (e.g., due to unknown elements).

5. **Save Results**:  
   Store the resulting DataFrame (with embeddings) as a pickle file for each dataset.

In [ ]:
from src.create_embeddings import create_embeddings

In [ ]:
create_embeddings()

### 3. Compress Embeddings with PCA

1. **Load Pre-Computed Embeddings**:  
   Reads compound embeddings from pickle files (for RE-free, RE-containing, and All experimental datasets).

2. **Apply PCA**:  
   Compresses high-dimensional embeddings (e.g., 200+ dims) into lower-dimensional representations using PCA with 8, 16, 32, and 64 components.

3. **Preserve Explained Variance**:  
   Tracks how much variance each reduced dimension captures (typically >90% with 64 components).

4. **Save Compressed Embeddings**:  
   Adds new columns (`comp_emb_pca_8`, `comp_emb_pca_16`, etc.) to the DataFrame and saves the result as a new pickle file.

In [ ]:
from src.compress_embeddings_pca import compress_embeddings_pca

In [ ]:
compress_embeddings_pca()

### 4. Model Training

1. **Input**:  
   Preprocessed simulation datasets with PCA-compressed compound embeddings (8–64D) and Tc_sim targets.

2. **Models Trained per Dataset** (RE-Free, RE, All):  
   - **Linear** (LassoLars, Ridge)  
   - **Random Forest**  
   - **MLP (Neural Network)**  
   All trained on raw 200D and PCA-reduced embeddings (8, 16, 32, 64D).

3. **Hyperparameter Tuning**:  
   - Randomized search (RF, MLP) and grid search (linear).  
   - Search space scaled to dataset size (e.g., fewer iterations for larger datasets).

4. **Evaluation & Visualization**:  
   - Metrics: R², MAE, RMSE.  
   - Plots: Predicted vs. actual Tc for train/test sets.

5. **Model Export**:  
   - Best models saved as ONNX (with preprocessing included).

6. **Output**:  
   - Per-dataset results (`results/<dataset>_results.csv`).  
   - Global comparison and best model summary (`sim_tc_comparison.csv`, `sim_tc_best_by_dataset.csv`).  
   - Figures and ONNX models for analysis and inference.